# Training models + résultats

Lightweight notebook to show only:
- model **training**
- **final results** (without unnecessary intermediate blocks)
> This notebook expects a `longitudinal_data.csv` file in the same folder.


In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

csv_path = "longitudinal_data.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        "Le fichier '.csv' est introuvable. "
        "Place-le dans le même dossier que ce notebook."
    )

df = pd.read_csv(csv_path)

# Harmonisation des noms de colonnes
df.rename(columns=lambda x: x.replace("J14", "D14"), inplace=True)
df.rename(columns={"CAG": "GSRC"}, inplace=True)

print("Dimensions du dataset :", df.shape)
print("\nColonnes disponibles :")
print(df.columns.tolist())


c:\Users\pc\miniconda3\lib\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Dimensions du dataset : (104, 24)

Colonnes disponibles :
['patient', 'GSRC', 'condition', 'Cadence_spm_D14', 'Cadence_spm_J56', 'Cadence_stps_D14', 'Cadence_stps_J56', 'EH_D14', 'EH_J56', 'FG_D14', 'FG_J56', 'FH_D14', 'FH_J56', 'LP_D14', 'LP_J56', 'V_D14', 'V_J56', 'Delta_V', 'Delta_LP', 'Delta_Cadence_stps', 'Delta_Cadence_spm', 'Delta_FG', 'Delta_FH', 'Delta_EH']


## 1) Definition of targets, features, and models

In [2]:

targets = ["Delta_V", "Delta_LP", "Delta_FG", "Delta_FH", "Delta_EH"]

features_base = ["FG_D14", "FH_D14", "EH_D14", "V_D14", "LP_D14", "Cadence_stps_D14"]
features_with_gsrc = features_base + ["GSRC"]

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("Targets :", targets)
print("Features sans GSRC :", features_base)
print("Features avec GSRC :", features_with_gsrc)


Targets : ['Delta_V', 'Delta_LP', 'Delta_FG', 'Delta_FH', 'Delta_EH']
Features sans GSRC : ['FG_D14', 'FH_D14', 'EH_D14', 'V_D14', 'LP_D14', 'Cadence_stps_D14']
Features avec GSRC : ['FG_D14', 'FH_D14', 'EH_D14', 'V_D14', 'LP_D14', 'Cadence_stps_D14', 'GSRC']


## 2) Training bloc

In [3]:

def clone_model(model):
    if isinstance(model, LinearRegression):
        return LinearRegression()
    if isinstance(model, Ridge):
        return Ridge(alpha=model.alpha)
    if isinstance(model, Lasso):
        return Lasso(alpha=model.alpha, max_iter=model.max_iter)
    if isinstance(model, RandomForestRegressor):
        return RandomForestRegressor(
            n_estimators=model.n_estimators,
            random_state=model.random_state
        )
    if isinstance(model, GradientBoostingRegressor):
        return GradientBoostingRegressor(
            random_state=model.random_state
        )
    raise ValueError("Modèle non supporté.")

def evaluate_model_cv(model, X, y, cv):
    mae_scores, rmse_scores, r2_scores = [], [], []

    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        current_model = clone_model(model)
        current_model.fit(X_train_scaled, y_train)
        y_pred = current_model.predict(X_test_scaled)

        mae_scores.append(mean_absolute_error(y_test, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_test, y_pred)))
        r2_scores.append(r2_score(y_test, y_pred))

    return {
        "MAE_mean": np.mean(mae_scores),
        "MAE_std": np.std(mae_scores),
        "RMSE_mean": np.mean(rmse_scores),
        "RMSE_std": np.std(rmse_scores),
        "R2_median": np.median(r2_scores),
        "R2_mean": np.mean(r2_scores),
        "R2_std": np.std(r2_scores)
    }

results_no_gsrc = []
results_with_gsrc = []

for target in targets:
    y = df[target]

    X_no = df[features_base]
    X_yes = df[features_with_gsrc]

    for model_name, model in models.items():
        scores_no = evaluate_model_cv(model, X_no, y, kf)
        scores_yes = evaluate_model_cv(model, X_yes, y, kf)

        results_no_gsrc.append({
            "Target": target,
            "Model": model_name,
            **scores_no
        })

        results_with_gsrc.append({
            "Target": target,
            "Model": model_name,
            **scores_yes
        })

results_no_gsrc = pd.DataFrame(results_no_gsrc).sort_values(["Target", "R2_median"], ascending=[True, False])
results_with_gsrc = pd.DataFrame(results_with_gsrc).sort_values(["Target", "R2_median"], ascending=[True, False])

print("Training terminé.")


Training terminé.


## 3) Complete results

In [4]:

print("=== RÉSULTATS SANS GSRC ===")
display(results_no_gsrc)

print("\n=== RÉSULTATS AVEC GSRC ===")
display(results_with_gsrc)


=== RÉSULTATS SANS GSRC ===


,Target,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_median,R2_mean,R2_std
22,Delta_EH,Lasso,34.052817,55.974889,135.495568,254.956863,0.394949,-1451.824068,2904.461526
20,Delta_EH,Linear,45.595524,79.063407,188.402331,360.769156,0.394917,-2886.355930,5773.524825
21,Delta_EH,Ridge,12.134967,12.125015,35.087374,54.128469,0.394545,-71.102311,143.017189
23,Delta_EH,RandomForest,6.150821,1.065960,8.104160,1.377945,0.307032,0.325138,0.097407
24,Delta_EH,GradientBoosting,6.793472,0.976831,8.818531,1.383766,0.240750,0.205823,0.061424
13,Delta_FG,RandomForest,4.451382,0.422798,5.773023,0.505170,-0.059447,0.011955,0.150410
11,Delta_FG,Ridge,17.969936,27.078287,66.461787,121.510001,-0.072388,-432.431490,864.869835
10,Delta_FG,Linear,40.290829,71.691521,168.796392,326.149477,-0.074368,-3045.319121,6090.635024
12,Delta_FG,Lasso,28.959509,49.039861,116.852089,222.272025,-0.075708,-1423.466605,2846.934153
14,Delta_FG,GradientBoosting,5.016989,0.495469,6.412543,0.634591,-0.298526,-0.214831,0.165276



=== RÉSULTATS AVEC GSRC ===


,Target,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_median,R2_mean,R2_std
21,Delta_EH,Ridge,11.837755,11.468879,33.659298,51.179910,0.384964,-63.972241,128.743203
22,Delta_EH,Lasso,34.078189,55.961330,135.529636,254.936399,0.384948,-1451.797757,2904.395935
20,Delta_EH,Linear,45.913821,79.627852,189.764966,363.397355,0.384361,-2928.513113,5857.824889
23,Delta_EH,RandomForest,6.133939,1.050330,8.089370,1.361382,0.311979,0.327575,0.094462
24,Delta_EH,GradientBoosting,6.928407,1.092449,9.099494,1.529888,0.133854,0.159103,0.034882
13,Delta_FG,RandomForest,4.418210,0.415697,5.750172,0.536510,-0.017514,0.018953,0.161302
11,Delta_FG,Ridge,19.574045,30.137262,73.743351,135.740724,-0.223575,-538.196271,1076.263217
14,Delta_FG,GradientBoosting,4.993419,0.468670,6.344577,0.597717,-0.235331,-0.189841,0.164492
12,Delta_FG,Lasso,32.904703,56.781008,134.860956,257.948779,-0.235927,-1912.840318,3825.541337
10,Delta_FG,Linear,44.676481,80.311903,188.810389,365.828046,-0.243799,-3827.143193,7654.139184


## 4) Best model by target

In [5]:

best_no_gsrc = results_no_gsrc.groupby("Target", as_index=False).first()
best_with_gsrc = results_with_gsrc.groupby("Target", as_index=False).first()

print("=== MEILLEUR MODÈLE PAR TARGET (sans GSRC) ===")
display(best_no_gsrc)

print("\n=== MEILLEUR MODÈLE PAR TARGET (avec GSRC) ===")
display(best_with_gsrc)


=== MEILLEUR MODÈLE PAR TARGET (sans GSRC) ===


,Target,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_median,R2_mean,R2_std
0,Delta_EH,Lasso,34.052817,55.974889,135.495568,254.956863,0.394949,-1451.824068,2904.461526
1,Delta_FG,RandomForest,4.451382,0.422798,5.773023,0.505170,-0.059447,0.011955,0.150410
2,Delta_FH,Ridge,12.087335,10.180640,30.396232,42.294355,0.407943,-26.757963,54.334473
3,Delta_LP,Lasso,0.042538,0.017256,0.076219,0.056024,-0.209898,-3.794899,7.382366
4,Delta_V,GradientBoosting,4.006455,3.602557,15.730348,13.823592,-0.091025,-37511.850265,59840.706401



=== MEILLEUR MODÈLE PAR TARGET (avec GSRC) ===


,Target,Model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_median,R2_mean,R2_std
0,Delta_EH,Ridge,11.837755,11.468879,33.659298,51.179910,0.384964,-63.972241,128.743203
1,Delta_FG,RandomForest,4.418210,0.415697,5.750172,0.536510,-0.017514,0.018953,0.161302
2,Delta_FH,Ridge,10.934054,7.887417,25.227675,31.885475,0.416138,-15.816616,32.445114
3,Delta_LP,Lasso,0.042540,0.017256,0.076208,0.056025,-0.209898,-3.794614,7.382517
4,Delta_V,GradientBoosting,4.098358,3.710265,16.044757,14.073187,-0.087778,-40643.372854,65554.416693


## 5) Direct comparison: does GSRC help?

In [6]:

comparison = best_no_gsrc.merge(
    best_with_gsrc,
    on="Target",
    suffixes=("_no_GSRC", "_with_GSRC")
)

comparison["Delta_R2_median"] = comparison["R2_median_with_GSRC"] - comparison["R2_median_no_GSRC"]
comparison["Delta_RMSE"] = comparison["RMSE_mean_with_GSRC"] - comparison["RMSE_mean_no_GSRC"]

comparison = comparison[[
    "Target",
    "Model_no_GSRC", "R2_median_no_GSRC", "RMSE_mean_no_GSRC",
    "Model_with_GSRC", "R2_median_with_GSRC", "RMSE_mean_with_GSRC",
    "Delta_R2_median", "Delta_RMSE"
]]

display(comparison.sort_values("Delta_R2_median", ascending=False))


,Target,Model_no_GSRC,R2_median_no_GSRC,RMSE_mean_no_GSRC,Model_with_GSRC,R2_median_with_GSRC,RMSE_mean_with_GSRC,Delta_R2_median,Delta_RMSE
1,Delta_FG,RandomForest,-0.059447,5.773023,RandomForest,-0.017514,5.750172,0.041933,-0.022850
2,Delta_FH,Ridge,0.407943,30.396232,Ridge,0.416138,25.227675,0.008195,-5.168557
4,Delta_V,GradientBoosting,-0.091025,15.730348,GradientBoosting,-0.087778,16.044757,0.003246,0.314408
3,Delta_LP,Lasso,-0.209898,0.076219,Lasso,-0.209898,0.076208,0.000000,-0.000011
0,Delta_EH,Lasso,0.394949,135.495568,Ridge,0.384964,33.659298,-0.009986,-101.836270


## 6) Export LaTeX Table

In [7]:

print("=== LATEX : meilleurs modèles sans GSRC ===")
print(best_no_gsrc.to_latex(index=False, float_format="%.3f"))

print("\n=== LATEX : meilleurs modèles avec GSRC ===")
print(best_with_gsrc.to_latex(index=False, float_format="%.3f"))

print("\n=== LATEX : comparaison GSRC vs sans GSRC ===")
print(comparison.to_latex(index=False, float_format="%.3f"))


=== LATEX : meilleurs modèles sans GSRC ===
\begin{tabular}{llrrrrrrr}
\toprule
Target & Model & MAE_mean & MAE_std & RMSE_mean & RMSE_std & R2_median & R2_mean & R2_std \\
\midrule
Delta_EH & Lasso & 34.053 & 55.975 & 135.496 & 254.957 & 0.395 & -1451.824 & 2904.462 \\
Delta_FG & RandomForest & 4.451 & 0.423 & 5.773 & 0.505 & -0.059 & 0.012 & 0.150 \\
Delta_FH & Ridge & 12.087 & 10.181 & 30.396 & 42.294 & 0.408 & -26.758 & 54.334 \\
Delta_LP & Lasso & 0.043 & 0.017 & 0.076 & 0.056 & -0.210 & -3.795 & 7.382 \\
Delta_V & GradientBoosting & 4.006 & 3.603 & 15.730 & 13.824 & -0.091 & -37511.850 & 59840.706 \\
\bottomrule
\end{tabular}


=== LATEX : meilleurs modèles avec GSRC ===
\begin{tabular}{llrrrrrrr}
\toprule
Target & Model & MAE_mean & MAE_std & RMSE_mean & RMSE_std & R2_median & R2_mean & R2_std \\
\midrule
Delta_EH & Ridge & 11.838 & 11.469 & 33.659 & 51.180 & 0.385 & -63.972 & 128.743 \\
Delta_FG & RandomForest & 4.418 & 0.416 & 5.750 & 0.537 & -0.018 & 0.019 & 0.161 \\
Delta_FH